# Multi-Disease Medical AI — Module 3: Retinal Fundus Diabetic Retinopathy Model
### ECE Major Project
**Modality:** Retinal Fundus Photography (RetinaMNIST / DeepDR / APTOS, $224 \times 224$)
**Objective:** 5-Stage Diabetic Retinopathy Staging (Grade 0: Normal to Grade 4: Proliferative DR) using ResNet-50 + Ben Graham DIP + Quadratic Weighted Kappa (QWK).

In [ ]:
!pip install -q medmnist torchvision scikit-learn opencv-python-headless matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models
from medmnist import RetinaMNIST, INFO
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Compute Device:", device)

In [ ]:
# RetinaMNIST Datasets (224x224)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = RetinaMNIST(split="train", download=True, size=224, transform=transform)
val_dataset   = RetinaMNIST(split="val", download=True, size=224, transform=transform)
test_dataset  = RetinaMNIST(split="test", download=True, size=224, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"RetinaMNIST: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")

## ResNet-50 Architecture for Retinopathy Staging

In [ ]:
model = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 5)  # 5 DR grades
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

In [ ]:
def evaluate_qwk(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.squeeze().numpy())
    acc = accuracy_score(all_targets, all_preds)
    qwk = cohen_kappa_score(all_targets, all_preds, weights="quadratic")
    cm = confusion_matrix(all_targets, all_preds)
    return acc, qwk, cm

## Grad-CAM Localization of Retinal Microaneurysms and Hemorrhages

In [ ]:
target_layer = model.layer4[-1].conv3
print("Grad-CAM Target Layer configured:", target_layer)
torch.save(model.state_dict(), "retina_resnet50.pth")
print("Retina model weights successfully saved.")